# HLCA — Manuscript-ready results (Table 1 + figures)

## Rationale (why this experiment exists)

The Human Lung Cell Atlas (HLCA) is a realistic integration setting where **identifier harmonization is a prerequisite** for atlas-scale model training.
This experiment is designed to market the *practical* value of IDTrack by turning harmonization into a **reportable, auditable artifact**:

- HLCA contains many studies/datasets that were processed under different Ensembl releases and annotation conventions.
- The same biological entity can appear under different identifiers, and conversely one identifier can legitimately expand to multiple candidates (splits/merges/history).
- IDTrack makes these outcomes explicit (1→0 / 1→1 / 1→n) and keeps the conversion reproducible via a snapshot-bounded graph.

## What this notebook produces (manuscript-facing)

- **Table 1 (canonical)**: `idtrack-manuscript/tables/hlca_harmonization.tex`
- **Extended diagnostics (optional)**: `idtrack-manuscript/tables/hlca_harmonization_extended.tex`
- **Figures (optional pool for the Results narrative)**:
  - `idtrack-manuscript/figures/fig_hlca_outcomes_hgnc.pdf`
  - `idtrack-manuscript/figures/fig_hlca_target_compare.pdf`
  - `idtrack-manuscript/figures/fig_hlca_gene_overlap.pdf`
  - `idtrack-manuscript/figures/fig_hlca_jaccard_heatmap.pdf`
  - `idtrack-manuscript/figures/fig_hlca_throughput_scaling.pdf`

## Reproducibility contract (what readers can rerun)

This notebook intentionally foregrounds the knobs you can report in a Methods section:

- **Graph snapshot boundary** (what Ensembl history window is considered)
- **Target release** (what time point you harmonize into)
- **Target namespace** (HGNC vs Ensembl backbone; both are reported)
- **Ambiguity policy** (`strategy='all'` vs `strategy='best'`)

## Cache-first execution (no "RUN_*" toggles)

- If conversion caches exist, the notebook skips graph work and only composes tables/figures.
- If caches are missing, the notebook computes them once and writes under `idtrack/docs/_notebooks/idtrack_cache/experiments/hlca/`.

## Inputs and environment variables

- `HLCA_BASE_PATH`: HLCA data root (required only if caches are missing).
- `IDTRACK_LOCAL_REPO`: IDTrack cache directory (recommended: `idtrack/docs/_notebooks/idtrack_cache`).

This notebook uses the same curated HLCA study→files mapping as:
- `idtrack/docs/_notebooks/05_tutorial_harmonization.ipynb`

## Interpretation guide (how to read Table 1)

- **1→0**: no resolvable target (lost identifiers; measurable and reportable).
- **1→1**: unambiguous resolution.
- **1→n**: legitimate ambiguity (splits/merges/history); not a software error.
- **TDM vs ATM** (HGNC target): ATM indicates an Ensembl-side fallback where HGNC lacked a synonym for the target.


In [ ]:
from __future__ import annotations

import os
import time
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import sys

# Add experiments/src to sys.path (works even when launched from nested folders)
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not ((REPO_ROOT / 'idtrack').is_dir() and (REPO_ROOT / 'idtrack-manuscript').is_dir()):
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENTS_SRC = REPO_ROOT / 'idtrack' / 'reproducibility' / 'experiments' / 'src'
sys.path.append(str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    apply_rcparams,
    atomic_write_text,
    experiments_cache_dir,
    idtrack_cache_dir,
    load_rcparams,
    manuscript_palette,
    manuscript_figures_dir,
    manuscript_tables_dir,
    read_pickle,
    write_pickle,
)

try:
    apply_rcparams(load_rcparams())
except Exception as e:  # noqa: S110
    print('Warning: could not apply shared rcParams:', e)

if sns is not None:
    sns.set_theme(style='whitegrid', context='paper')

plt.rcParams.update({'savefig.dpi': 300, 'figure.dpi': 140})

print('Repo root:', REPO_ROOT)


In [ ]:
# -------------------- Configuration --------------------

# HLCA data root.
# Recommended: export in your shell so you never edit notebooks.
#   export HLCA_BASE_PATH=/path/to/HLCA_reproducibility/data
DEFAULT_HLCA_BASE_PATH = ''

base_path = os.environ.get('HLCA_BASE_PATH', DEFAULT_HLCA_BASE_PATH).strip()
HLCA_BASE_PATH = Path(base_path).expanduser().resolve() if base_path else None

# Shared IDTrack cache folder (graphs + databases + other heavy artefacts).
# Recommended for this repo: `idtrack/docs/_notebooks/idtrack_cache`
IDTRACK_LOCAL_REPO = idtrack_cache_dir(REPO_ROOT)

# Experiment cache (conversion pickles + derived CSVs).
HLCA_CACHE = experiments_cache_dir(REPO_ROOT, experiment='hlca')
CONVERSIONS_DIR = (HLCA_CACHE / 'conversions')
CONVERSIONS_DIR.mkdir(parents=True, exist_ok=True)

# Manuscript output locations
MANUSCRIPT_TABLES = manuscript_tables_dir(REPO_ROOT)
MANUSCRIPT_FIGURES = manuscript_figures_dir(REPO_ROOT)

# Manuscript choices
TARGET_RELEASE = 107
CONVERSION_STRATEGY = 'all'  # keep 'all' to expose 1→n explicitly

# Graph snapshot boundary used when conversions must be (re)computed.
# Leave as None to use the latest release reported by Ensembl REST.
GRAPH_SNAPSHOT_RELEASE: int | None = None

# Targets to evaluate.
# - `None` means: stay on Ensembl gene backbone (target IDs are Ensembl genes).
# - External DB names (e.g. 'HGNC Symbol') request final conversion into that namespace.
FINAL_DATABASES: dict[str, str | None] = {
    'ensembl_gene': None,
    'HGNC Symbol': 'HGNC Symbol',
}

print('HLCA_BASE_PATH:', HLCA_BASE_PATH)
print('IDTRACK_LOCAL_REPO:', IDTRACK_LOCAL_REPO)
print('HLCA_CACHE:', HLCA_CACHE)
print('CONVERSIONS_DIR:', CONVERSIONS_DIR)
print('MANUSCRIPT_TABLES:', MANUSCRIPT_TABLES)
print('MANUSCRIPT_FIGURES:', MANUSCRIPT_FIGURES)


In [ ]:
# -------------------- Curated HLCA study→files mapping --------------------

# Mirrors `idtrack/docs/_notebooks/05_tutorial_harmonization.ipynb`.
# If your HLCA directory structure differs, edit the two dataset directories below.

if HLCA_BASE_PATH is None:
    print('Set HLCA_BASE_PATH to your HLCA data root to enable this notebook.')

DEFAULT_DSET0_REL = 'HLCA_extended/extension_datasets/ready/full'
DEFAULT_DSET1_REL = 'HLCA_extended/extension_datasets/raw'

# These reflect the standard HLCA reproducibility layout.
dset0_dir = (HLCA_BASE_PATH / DEFAULT_DSET0_REL) if HLCA_BASE_PATH else None
dset1_dir = (HLCA_BASE_PATH / DEFAULT_DSET1_REL) if HLCA_BASE_PATH else None

hlca_adata_dict: dict[str, list[str]] = {}

if dset0_dir and dset1_dir:
    hlca_adata_dict = {
        'Kaminski_2020': [f'{dset0_dir}/adams.h5ad'],
        'Meyer_2021': [f'{dset0_dir}/meyer_2021.h5ad'],
        'MeyerNikolic_unpubl': [f'{dset0_dir}/meyer_nikolic_unpubl.h5ad'],
        'Barbry_unpubl': [f'{dset0_dir}/barbry.h5ad'],
        'Regev_2021': [
            f'{dset0_dir}/delorey_cryo.h5ad',
            f'{dset0_dir}/delorey_fresh.h5ad',
            f'{dset0_dir}/delorey_nuclei.h5ad',
        ],
        'Thienpont_2018': [f'{dset1_dir}/Lambrechts/lambrechts.h5ad'],
        'Budinger_2020': [f'{dset0_dir}/bharat.h5ad'],
        'Banovich_Kropski_2020': [f'{dset0_dir}/haberman.h5ad'],
        'Sheppard_2020': [f'{dset0_dir}/tsukui.h5ad'],
        'Wunderink_2021': [f'{dset0_dir}/grant_cryo.h5ad', f'{dset0_dir}/grant_fresh.h5ad'],
        'Lambrechts_2021': [f'{dset0_dir}/wouters.h5ad'],
        'Zhang_2021': [f'{dset1_dir}/Liao/covid_for_publish.h5ad'],
        'Duong_lungMAP_unpubl': [f'{dset0_dir}/duong.h5ad'],
        'Janssen_2020': [f'{dset0_dir}/mould.h5ad'],
        'Sun_2020': [
            f'{dset0_dir}/wang_sub_batch1.h5ad',
            f'{dset0_dir}/wang_sub_batch2.h5ad',
            f'{dset0_dir}/wang_sub_batch3.h5ad',
            f'{dset0_dir}/wang_sub_batch4.h5ad',
        ],
        'Gomperts_2021': [
            f'{dset0_dir}/carraro_ucla.h5ad',
            f'{dset0_dir}/carraro_cff.h5ad',
            f'{dset0_dir}/carraro_csmc.h5ad',
        ],
        'Eils_2020': [f'{dset0_dir}/lukassen.h5ad'],
        'Schiller_2020': [f'{dset0_dir}/mayr.h5ad'],
        'Misharin_Budinger_2018': [f'{dset0_dir}/reyfman_disease.h5ad'],
        'Shalek_2018': [f'{dset0_dir}/ordovasmontanes.h5ad'],
        'Schiller_2021': [f'{dset0_dir}/schiller_discovair.h5ad'],
        'Peer_Massague_2020': [f'{dset0_dir}/laughney.h5ad'],
        'Lafyatis_2019': [f'{dset0_dir}/valenzi.h5ad'],
        'Tata_unpubl': [f'{dset0_dir}/tata_unpubl.h5ad'],
        'Xu_2020': [f'{dset0_dir}/guo.h5ad'],
        'Sims_2019': [f'{dset0_dir}/szabo.h5ad'],
        'Schultze_unpubl': [f'{dset0_dir}/schultze.h5ad'],
    }

print('Studies configured:', len(hlca_adata_dict))
print('Example:', list(hlca_adata_dict)[:5])


In [ ]:
# -------------------- Validate file availability --------------------

flat_rows: list[tuple[str, str, bool]] = []
missing_rows: list[tuple[str, str]] = []

for study, paths in hlca_adata_dict.items():
    for p in paths:
        exists = Path(p).exists()
        flat_rows.append((study, p, exists))
        if not exists:
            missing_rows.append((study, p))

hlca_file_status = pd.DataFrame(flat_rows, columns=['study', 'path', 'exists'])

if hlca_file_status.empty:
    print('No HLCA file mapping available (HLCA_BASE_PATH unset or directory layout not found).')
else:
    print('Total referenced .h5ad files:', int(hlca_file_status.shape[0]))
    print('Total found on disk:', int(hlca_file_status['exists'].sum()))

    if missing_rows:
        print()
        print(f"Missing files referenced by the curated list: {len(missing_rows)}")
        print('Showing first 15 missing:')
        for study, p in missing_rows[:15]:
            print(f' - {study}: {p}')

if hlca_file_status.empty:
    hlca_file_status
else:
    (
        hlca_file_status.groupby('study')['exists']
        .agg(['count', 'sum'])
        .rename(columns={'count': 'n_files', 'sum': 'n_found'})
        .sort_values(['n_found', 'n_files'], ascending=False)
    )


In [ ]:
# -------------------- Conversion cache I/O --------------------


def _safe_stem(s: str) -> str:
    return ''.join(c if c.isalnum() or c in {'-', '_'} else '_' for c in str(s))


def _final_label(final_database: str | None) -> str:
    return 'ensembl_gene' if final_database is None else str(final_database)


def _pickle_path(study: str, *, final_database: str | None) -> Path:
    tag = (
        f"hlca_{_safe_stem(study)}_toRelease{TARGET_RELEASE}_final{_safe_stem(_final_label(final_database))}"
        f"_strategy{CONVERSION_STRATEGY}.pickle"
    )
    return CONVERSIONS_DIR / tag


@dataclass(frozen=True)
class ConversionPayload:
    matchings: list[dict]
    seconds: float
    n_inputs: int
    graph_snapshot_release: int | None
    target_release: int
    final_database: str | None
    strategy: str

    @property
    def it_per_s(self) -> float:
        return (self.n_inputs / self.seconds) if self.seconds else float('nan')


def _load_payload(study: str, *, final_database: str | None) -> ConversionPayload | None:
    p = _pickle_path(study, final_database=final_database)
    if not p.exists():
        return None

    obj = read_pickle(p)

    # Backwards-compat: some earlier caches stored just `list[dict]`.
    if isinstance(obj, list):
        return ConversionPayload(
            matchings=obj,
            seconds=float('nan'),
            n_inputs=len(obj),
            graph_snapshot_release=None,
            target_release=TARGET_RELEASE,
            final_database=final_database,
            strategy=CONVERSION_STRATEGY,
        )

    if not isinstance(obj, dict) or 'matchings' not in obj:
        raise TypeError(f"Unexpected payload in {p}: expected dict with 'matchings'.")

    matchings = obj['matchings']
    return ConversionPayload(
        matchings=matchings,
        seconds=float(obj.get('seconds', float('nan'))),
        n_inputs=int(obj.get('n_inputs', len(matchings))),
        graph_snapshot_release=(int(obj['graph_snapshot_release']) if obj.get('graph_snapshot_release') is not None else None),
        target_release=int(obj.get('target_release', TARGET_RELEASE)),
        final_database=obj.get('final_database', final_database),
        strategy=str(obj.get('strategy', CONVERSION_STRATEGY)),
    )


def _save_payload(study: str, *, final_database: str | None, payload: ConversionPayload) -> Path:
    p = _pickle_path(study, final_database=final_database)
    return write_pickle(
        {
            'matchings': payload.matchings,
            'seconds': payload.seconds,
            'n_inputs': payload.n_inputs,
            'graph_snapshot_release': payload.graph_snapshot_release,
            'target_release': payload.target_release,
            'final_database': payload.final_database,
            'strategy': payload.strategy,
        },
        p,
    )


In [ ]:
# -------------------- Ensure conversion caches exist --------------------


def _studies_with_cached_conversions() -> set[str]:
    studies = set()
    for p in CONVERSIONS_DIR.glob('hlca_*_toRelease*_final*_strategy*.pickle'):
        name = p.name
        if not name.startswith('hlca_'):
            continue
        # hlca_<study>_toRelease...
        mid = name[len('hlca_'):]
        study = mid.split('_toRelease', 1)[0]
        if study:
            studies.add(study)
    return studies


def _studies_with_h5ad_files() -> list[str]:
    if hlca_file_status.empty:
        return []
    return [s for s, grp in hlca_file_status.groupby('study') if grp['exists'].any()]


def _read_var_names_backed(h5ad_path: str) -> list[str]:
    import anndata as ad

    adata = ad.read_h5ad(h5ad_path, backed='r')
    try:
        return adata.var_names.astype(str).tolist()
    finally:
        try:
            adata.file.close()
        except Exception:
            pass


def _study_union_gene_list(study: str) -> list[str]:
    genes: set[str] = set()
    for p in hlca_adata_dict.get(study, []):
        if Path(p).exists():
            genes.update(_read_var_names_backed(p))
    return sorted(genes)


# Decide which studies to process.
# - If HLCA paths are available, we use the curated mapping.
# - Otherwise, we rely on whatever is already cached on disk.
studies = _studies_with_h5ad_files() or sorted(_studies_with_cached_conversions())

missing_jobs: list[tuple[str, str, str | None]] = []
for study in studies:
    for label, final_db in FINAL_DATABASES.items():
        if _load_payload(study, final_database=final_db) is None:
            missing_jobs.append((study, label, final_db))

if not studies:
    print('No HLCA studies detected (no HLCA paths and no cached conversions).')
elif not missing_jobs:
    print(f'All conversion caches present for {len(studies)} studies; skipping IDTrack graph load.')
else:
    if not hlca_adata_dict:
        raise RuntimeError(
            'Conversion caches are missing, but HLCA_BASE_PATH is not configured. '            'Set HLCA_BASE_PATH so the notebook can read HLCA .h5ad var_names and compute caches.'
        )

    try:
        import idtrack
    except ImportError as e:
        raise ImportError('Computing HLCA caches requires `idtrack` installed.') from e

    # Build graph once.
    api = idtrack.API(local_repository=str(IDTRACK_LOCAL_REPO))
    api.configure_logger()

    organism, latest_release = api.resolve_organism('human')
    snapshot = int(GRAPH_SNAPSHOT_RELEASE) if GRAPH_SNAPSHOT_RELEASE is not None else int(latest_release)
    snapshot = max(snapshot, int(TARGET_RELEASE))

    print(f'Building/loading IDTrack graph for {organism} snapshot_release={snapshot} (target_release={TARGET_RELEASE})')
    api.build_graph(organism_name=organism, snapshot_release=snapshot, calculate_caches=True)

    # Cache union gene lists per study to avoid re-reading .h5ad files.
    gene_lists: dict[str, list[str]] = {}

    runs = []
    for study, label, final_db in missing_jobs:
        if study not in gene_lists:
            gene_lists[study] = _study_union_gene_list(study)

        ids = gene_lists[study]
        if not ids:
            print('Skip (no genes found):', study)
            continue

        t0 = time.perf_counter()
        matchings = api.convert_identifier_multiple(
            ids,
            to_release=int(TARGET_RELEASE),
            final_database=final_db,
            strategy=CONVERSION_STRATEGY,
            verbose=True,
            pbar_prefix=f"HLCA:{study}:{label}",
        )
        dt = time.perf_counter() - t0

        payload = ConversionPayload(
            matchings=matchings,
            seconds=dt,
            n_inputs=len(ids),
            graph_snapshot_release=snapshot,
            target_release=int(TARGET_RELEASE),
            final_database=final_db,
            strategy=CONVERSION_STRATEGY,
        )

        out_p = _save_payload(study, final_database=final_db, payload=payload)
        runs.append(
            {
                'study': study,
                'label': label,
                'final_database': final_db if final_db is not None else 'ensembl_gene',
                'n_inputs': payload.n_inputs,
                'seconds': payload.seconds,
                'it_per_s': payload.it_per_s,
                'graph_snapshot_release': snapshot,
                'target_release': int(TARGET_RELEASE),
                'strategy': CONVERSION_STRATEGY,
            }
        )
        print('Saved:', out_p)

    if runs:
        timings = pd.DataFrame(runs).sort_values(['label', 'study']).reset_index(drop=True)
        timings_path = HLCA_CACHE / 'timings.csv'
        atomic_write_text(timings_path, timings.to_csv(index=False))
        print('Wrote timings:', timings_path)
        timings
    else:
        print('No caches were written (nothing to run).')


In [ ]:
# -------------------- Build Table 1 (from cached conversions) --------------------

from collections import Counter

try:
    import idtrack
except ImportError as e:
    raise ImportError('This analysis requires `idtrack` installed. Graph loading is only needed if caches are missing.') from e

api = idtrack.API(local_repository=str(IDTRACK_LOCAL_REPO))

# Load timings if available
_timings_path = HLCA_CACHE / 'timings.csv'
if _timings_path.exists():
    timings = pd.read_csv(_timings_path)
else:
    timings = pd.DataFrame(columns=['study', 'label', 'it_per_s'])


def _timing_lookup(study: str, label: str) -> float:
    if timings.empty:
        return float('nan')
    sub = timings[(timings['study'] == study) & (timings['label'] == label)]
    if sub.empty:
        return float('nan')
    return float(sub.iloc[0]['it_per_s'])


def _summarize_matchings(matchings: list[dict]) -> dict[str, int]:
    bins = api.classify_multiple_conversion(matchings)

    # 1→0 / 1→1 / 1→n
    n_inputs = len(bins['input_identifiers'])
    n_1to0 = len(bins['matching_1_to_0'])
    n_1to1 = len(bins['matching_1_to_1'])
    n_1ton = len(bins['matching_1_to_n'])

    # Diagnostics
    n_changed_1to1 = len(bins['changed_only_1_to_1'])
    n_changed_1ton = len(bins['changed_only_1_to_n'])
    n_fallback_1to1 = len(bins['alternative_target_1_to_1'])
    n_fallback_1ton = len(bins['alternative_target_1_to_n'])

    # n→1 collapses among 1→1 targets (important for harmonization)
    t1 = []
    for rec in bins['matching_1_to_1']:
        tid = rec.get('target_id', [])
        if isinstance(tid, list) and len(tid) == 1:
            t1.append(str(tid[0]))
    n_to_1_targets = sum(1 for _k, v in Counter(t1).items() if v > 1)

    return {
        'input_ids': n_inputs,
        'one_to_none': n_1to0,
        'one_to_one': n_1to1,
        'one_to_many': n_1ton,
        'changed_only_1_to_1': n_changed_1to1,
        'changed_only_1_to_n': n_changed_1ton,
        'fallback_1_to_1': n_fallback_1to1,
        'fallback_1_to_n': n_fallback_1ton,
        'n_to_1_targets_within_1_to_1': n_to_1_targets,
    }


def _targets_1to1(matchings: list[dict]) -> set[str]:
    out = set()
    bins = api.classify_multiple_conversion(matchings)
    for rec in bins['matching_1_to_1']:
        tid = rec.get('target_id', [])
        if isinstance(tid, list) and len(tid) == 1:
            out.add(str(tid[0]))
    return out


# Determine studies from either curated HLCA mapping or cached conversions.
if hlca_file_status.empty:
    studies = sorted({s for s in _studies_with_cached_conversions()})
else:
    studies = [s for s, grp in hlca_file_status.groupby('study') if grp['exists'].any()]

rows = []
missing = []

# For gene-overlap diagnostics (use Ensembl backbone targets)
per_study_ensembl_targets_1to1: dict[str, set[str]] = {}

for study in studies:
    p_hgnc = _load_payload(study, final_database='HGNC Symbol')
    p_ens = _load_payload(study, final_database=None)

    if p_hgnc is None or p_ens is None:
        missing.append(study)
        continue

    c_hgnc = _summarize_matchings(p_hgnc.matchings)
    c_ens = _summarize_matchings(p_ens.matchings)

    if c_hgnc['one_to_none'] != c_ens['one_to_none']:
        print('Warning: 1→0 differs between targets for', study, c_hgnc['one_to_none'], c_ens['one_to_none'])

    per_study_ensembl_targets_1to1[study] = _targets_1to1(p_ens.matchings)

    rows.append(
        {
            'Dataset': study,
            'Input IDs': c_ens['input_ids'],
            # HGNC target (TDM vs ATM = fallback)
            'HGNC 1→1 TDM': c_hgnc['one_to_one'],
            'HGNC 1→1 ATM': c_hgnc['fallback_1_to_1'],
            'HGNC 1→n TDM': c_hgnc['one_to_many'],
            'HGNC 1→n ATM': c_hgnc['fallback_1_to_n'],
            # Ensembl target (stay on backbone)
            'Ensembl 1→1': c_ens['one_to_one'],
            'Ensembl 1→n': c_ens['one_to_many'],
            '1→0': c_ens['one_to_none'],
            # Extra diagnostics (optional tables/figures)
            'Changed-only 1→1 (Ensembl)': c_ens['changed_only_1_to_1'],
            'Changed-only 1→n (Ensembl)': c_ens['changed_only_1_to_n'],
            'n→1 targets within 1→1 (Ensembl)': c_ens['n_to_1_targets_within_1_to_1'],
            # Speed
            'Performance (it/s) Ensembl IDs': _timing_lookup(study, 'ensembl_gene'),
            'Performance (it/s) HGNC Symbols': _timing_lookup(study, 'HGNC Symbol'),
        }
    )

if missing:
    print('Missing cached conversions for studies:', missing)

summary = pd.DataFrame(rows).sort_values('Dataset').reset_index(drop=True)

summary_path = HLCA_CACHE / 'hlca_summary.csv'
atomic_write_text(summary_path, summary.to_csv(index=False))
print('Wrote:', summary_path)

summary

# -------------------- Cross-dataset overlap diagnostics (Ensembl 1→1 only) --------------------

if per_study_ensembl_targets_1to1:
    counter = Counter()
    for s, genes in per_study_ensembl_targets_1to1.items():
        for g in genes:
            counter[g] += 1

    n_datasets = len(per_study_ensembl_targets_1to1)
    overlap = Counter(counter.values())

    overlap_df = (
        pd.DataFrame({'n_datasets_present': list(overlap.keys()), 'n_genes': list(overlap.values())})
        .sort_values('n_datasets_present')
        .reset_index(drop=True)
    )

    overlap_path = HLCA_CACHE / 'hlca_gene_overlap_distribution.csv'
    atomic_write_text(overlap_path, overlap_df.to_csv(index=False))
    print('Wrote:', overlap_path)

    core = int(overlap.get(n_datasets, 0))
    union = int(sum(overlap.values()))
    print(f'Ensembl (1→1 only) union genes: {union:,}')
    print(f'Ensembl (1→1 only) intersection genes (present in all {n_datasets} datasets): {core:,}')

    overlap_df
else:
    print('No per-study Ensembl 1→1 targets available; overlap diagnostics skipped.')

# -------------------- Pairwise overlap matrix (Ensembl 1→1 only) --------------------

# This supports a manuscript-friendly heatmap that markets "shared feature-space structure".
# The computation is cacheable and uses only the per-study 1→1 target sets extracted above.

if per_study_ensembl_targets_1to1:
    studies = sorted(per_study_ensembl_targets_1to1)
    jaccard = pd.DataFrame(index=studies, columns=studies, dtype=float)

    for s1 in studies:
        a = set(per_study_ensembl_targets_1to1[s1])
        for s2 in studies:
            b = set(per_study_ensembl_targets_1to1[s2])
            denom = len(a | b)
            jaccard.loc[s1, s2] = (len(a & b) / denom) if denom else float('nan')

    jaccard_path = HLCA_CACHE / 'hlca_jaccard_matrix.csv'
    atomic_write_text(jaccard_path, jaccard.to_csv())
    print('Wrote:', jaccard_path)


In [ ]:
# -------------------- Export LaTeX tables --------------------

out_tex_main = MANUSCRIPT_TABLES / 'hlca_harmonization.tex'
out_tex_ext = MANUSCRIPT_TABLES / 'hlca_harmonization_extended.tex'


def _latex_escape(value: str) -> str:
    s = str(value)
    s = s.replace('\\', r'\textbackslash{}')
    s = s.replace('&', r'\&')
    s = s.replace('%', r'\%')
    s = s.replace('_', r'\_')
    s = s.replace('#', r'\#')
    s = s.replace('{', r'\{')
    s = s.replace('}', r'\}')
    s = s.replace('^', r'\textasciicircum{}')
    s = s.replace('~', r'\textasciitilde{}')
    return s


def _fmt_int(x) -> str:
    try:
        return f"{int(x)}"
    except Exception:
        return str(x)


def _fmt_float(x, digits: int = 2) -> str:
    try:
        v = float(x)
        return f"{v:.{digits}f}" if v == v else ''
    except Exception:
        return ''


if summary.empty:
    print('No rows to export. Are HLCA conversions cached?')
else:
    caption = (
        'Summary of Human Lung Cell Atlas (HLCA) identifier harmonization performed with \textit{IDTrack}. '
        'All datasets are mapped into Ensembl release 107. The table reports explicit 1→0 / 1→1 / 1→n outcomes '
        'and distinguishes target-database matches from Ensembl fallbacks when the target database lacks a synonym.'
    )

    cols_main = [
        'Dataset',
        'Input IDs',
        'HGNC 1→1 TDM',
        'HGNC 1→1 ATM',
        'HGNC 1→n TDM',
        'HGNC 1→n ATM',
        'Ensembl 1→1',
        'Ensembl 1→n',
        '1→0',
        'Performance (it/s) Ensembl IDs',
        'Performance (it/s) HGNC Symbols',
    ]

    lines = []
    lines.append(r'\begin{table*}[t]')
    lines.append(r'\caption{')
    lines.append('    ' + caption)
    lines.append(r'}')
    lines.append(r'\label{tab:hlca}')
    lines.append(r'\centering')
    lines.append(r'\footnotesize')
    lines.append(r'\setlength{\tabcolsep}{3pt}')
    lines.append(r'\renewcommand{\arraystretch}{1.15}')
    lines.append(r'\begin{tabular}{@{}l r r r r r r r r r r@{}}')
    lines.append(r'\toprule')
    lines.append(r'& & \multicolumn{4}{c}{Target HGNC Symbols} & \multicolumn{2}{c}{Target Ensembl genes} & & \multicolumn{2}{c}{Performance (\textit{it/s})} \\')
    lines.append(r'\cmidrule(lr){3-6} \cmidrule(lr){7-8} \cmidrule(lr){10-11}')
    lines.append(
        r'Dataset & \shortstack[c]{Input\\IDs} & \multicolumn{2}{c}{\shortstack[c]{One-to-\\One}} & '
        r'\multicolumn{2}{c}{\shortstack[c]{One-to-\\Many}} & '
        r'\shortstack[c]{One-to-\\One} & \shortstack[c]{One-to-\\Many} & '
        r'\shortstack[c]{One-to-\\None} & '
        r'\shortstack[c]{Ensembl\\IDs} & \shortstack[c]{HGNC\\Symbols} \\'
    )
    lines.append(r'\cmidrule(lr){3-4} \cmidrule(lr){5-6}')
    lines.append(r'& & TDM & ATM & TDM & ATM \\')
    lines.append(r'\midrule')

    for _, r in summary[cols_main].iterrows():
        ds = _latex_escape(r['Dataset'])
        row = [
            f"\\textit{{{ds}}}",
            _fmt_int(r['Input IDs']),
            _fmt_int(r['HGNC 1→1 TDM']),
            _fmt_int(r['HGNC 1→1 ATM']),
            _fmt_int(r['HGNC 1→n TDM']),
            _fmt_int(r['HGNC 1→n ATM']),
            _fmt_int(r['Ensembl 1→1']),
            _fmt_int(r['Ensembl 1→n']),
            _fmt_int(r['1→0']),
            _fmt_float(r['Performance (it/s) Ensembl IDs']),
            _fmt_float(r['Performance (it/s) HGNC Symbols']),
        ]
        lines.append(' & '.join(row) + r' \\')

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table*}')

    atomic_write_text(out_tex_main, '\n'.join(lines) + '\n')
    print('Wrote:', out_tex_main)

    # Extended alternative (adds drift + collision diagnostics)
    cols_ext = cols_main + [
        'Changed-only 1→1 (Ensembl)',
        'Changed-only 1→n (Ensembl)',
        'n→1 targets within 1→1 (Ensembl)',
    ]

    ext = summary[cols_ext].copy()
    ext_path_csv = HLCA_CACHE / 'hlca_harmonization_extended.csv'
    atomic_write_text(ext_path_csv, ext.to_csv(index=False))
    print('Wrote:', ext_path_csv)

    lines2 = []
    lines2.append(r'\begin{table*}[t]')
    lines2.append(r'\caption{Extended HLCA harmonization diagnostics.}')
    lines2.append(r'\label{tab:hlca_extended}')
    lines2.append(r'\centering')
    lines2.append(r'\footnotesize')
    lines2.append(r'\setlength{\tabcolsep}{3pt}')
    lines2.append(r'\renewcommand{\arraystretch}{1.15}')
    lines2.append(r'\begin{tabular}{@{}l r r r r r r r r r r r@{}}')
    lines2.append(r'\toprule')
    lines2.append(r'Dataset & Input & HGNC 1→1 & HGNC 1→n & Ensembl 1→1 & Ensembl 1→n & 1→0 & Drift 1→1 & Drift 1→n & n→1 targets & it/s Ens & it/s HGNC \\')
    lines2.append(r'\midrule')

    for _, r in ext.iterrows():
        ds = _latex_escape(r['Dataset'])
        lines2.append(
            ' & '.join(
                [
                    f"\\textit{{{ds}}}",
                    _fmt_int(r['Input IDs']),
                    _fmt_int(r['HGNC 1→1 TDM'] + r['HGNC 1→1 ATM']),
                    _fmt_int(r['HGNC 1→n TDM'] + r['HGNC 1→n ATM']),
                    _fmt_int(r['Ensembl 1→1']),
                    _fmt_int(r['Ensembl 1→n']),
                    _fmt_int(r['1→0']),
                    _fmt_int(r['Changed-only 1→1 (Ensembl)']),
                    _fmt_int(r['Changed-only 1→n (Ensembl)']),
                    _fmt_int(r['n→1 targets within 1→1 (Ensembl)']),
                    _fmt_float(r['Performance (it/s) Ensembl IDs']),
                    _fmt_float(r['Performance (it/s) HGNC Symbols']),
                ]
            )
            + r' \\'
        )

    lines2.append(r'\bottomrule')
    lines2.append(r'\end{tabular}')
    lines2.append(r'\end{table*}')

    atomic_write_text(out_tex_ext, '\n'.join(lines2) + '\n')
    print('Wrote:', out_tex_ext)


In [ ]:
# -------------------- Manuscript-style plots --------------------

import numpy as np

if summary.empty:
    print('No summary available; skipping plots.')
else:
    df = summary.set_index('Dataset')

    # 1) Outcome profile per dataset (HGNC target)
    frac_hgnc = df[[
        'HGNC 1→1 TDM', 'HGNC 1→1 ATM', 'HGNC 1→n TDM', 'HGNC 1→n ATM', '1→0'
    ]].div(df['Input IDs'], axis=0)

    frac_hgnc = frac_hgnc.sort_values('1→0', ascending=False)

    colors = {
        'HGNC 1→1 TDM': MANUSCRIPT_COLORS['HGNC 1→1 TDM'],
        'HGNC 1→1 ATM': MANUSCRIPT_COLORS['HGNC 1→1 ATM'],
        'HGNC 1→n TDM': MANUSCRIPT_COLORS['HGNC 1→n TDM'],
        'HGNC 1→n ATM': MANUSCRIPT_COLORS['HGNC 1→n ATM'],
        '1→0': MANUSCRIPT_COLORS['1→0'],
    }

    fig, ax = plt.subplots(1, 1, figsize=(10, max(3.5, 0.25 * len(frac_hgnc))))

    left = None
    for col in frac_hgnc.columns:
        ax.barh(frac_hgnc.index, frac_hgnc[col], left=left, label=col, color=colors.get(col))
        left = frac_hgnc[col] if left is None else (left + frac_hgnc[col])

    ax.set_xlabel('Fraction of input IDs')
    ax.set_ylabel('HLCA dataset')
    ax.set_xlim(0, 1)
    ax.legend(loc='lower right', frameon=True)
    ax.set_title('HLCA identifier harmonization outcomes (target: HGNC symbols)')

    fig.tight_layout()
    out_fig = MANUSCRIPT_FIGURES / 'fig_hlca_outcomes_hgnc.pdf'
    fig.savefig(out_fig, bbox_inches='tight')
    print('Saved:', out_fig)

    # 2) Target comparison: ambiguity + failure fraction (HGNC vs Ensembl)
    frac_targets = pd.DataFrame(
        {
            'HGNC 1→n (total)': (df['HGNC 1→n TDM'] + df['HGNC 1→n ATM']) / df['Input IDs'],
            'Ensembl 1→n': df['Ensembl 1→n'] / df['Input IDs'],
            '1→0': df['1→0'] / df['Input IDs'],
        }
    ).sort_values('1→0', ascending=False)

    fig2, ax2 = plt.subplots(1, 1, figsize=(10, max(3.5, 0.25 * len(frac_targets))))
    frac_targets[['HGNC 1→n (total)', 'Ensembl 1→n', '1→0']].plot(
        kind='barh',
        ax=ax2,
        color=['#DD8452', '#4C72B0', '#C44E52'],
    )
    ax2.set_xlim(0, 1)
    ax2.set_xlabel('Fraction of input IDs')
    ax2.set_ylabel('HLCA dataset')
    ax2.set_title('Ambiguity and failure rates by target namespace')
    ax2.legend(loc='lower right', frameon=True)
    fig2.tight_layout()

    out_fig2 = MANUSCRIPT_FIGURES / 'fig_hlca_target_compare.pdf'
    fig2.savefig(out_fig2, bbox_inches='tight')
    print('Saved:', out_fig2)

    # 3) Cross-dataset overlap distribution (Ensembl 1→1 only)
    overlap_path = HLCA_CACHE / 'hlca_gene_overlap_distribution.csv'
    if overlap_path.exists():
        overlap_df = pd.read_csv(overlap_path)

        fig3, ax3 = plt.subplots(1, 1, figsize=(6.5, 4))
        ax3.bar(overlap_df['n_datasets_present'], overlap_df['n_genes'], color='#4C72B0')
        ax3.set_xlabel('# datasets a gene is present in (after 1→1 mapping)')
        ax3.set_ylabel('# genes')
        ax3.set_title('HLCA shared feature-space structure (Ensembl 1→1 only)')
        fig3.tight_layout()

        out_fig3 = MANUSCRIPT_FIGURES / 'fig_hlca_gene_overlap.pdf'
        fig3.savefig(out_fig3, bbox_inches='tight')
        print('Saved:', out_fig3)
    else:
        print('No overlap CSV found; skipping gene-overlap plot:', overlap_path)

    # 4) Throughput scaling (marketing: atlas-scale practicality)
    perf = summary[['Dataset', 'Input IDs', 'Performance (it/s) Ensembl IDs', 'Performance (it/s) HGNC Symbols']].copy()
    perf = perf.dropna()

    if not perf.empty:
        fig4, ax4 = plt.subplots(1, 1, figsize=(6.8, 4.2))
        ax4.scatter(
            perf['Input IDs'],
            perf['Performance (it/s) Ensembl IDs'],
            label='Target: Ensembl gene IDs',
            color=MANUSCRIPT_COLORS['1→1'],
            s=35,
            alpha=0.9,
        )
        ax4.scatter(
            perf['Input IDs'],
            perf['Performance (it/s) HGNC Symbols'],
            label='Target: HGNC symbols',
            color=MANUSCRIPT_COLORS['1→n'],
            s=35,
            alpha=0.9,
        )
        ax4.set_xscale('log')
        ax4.set_yscale('log')
        ax4.set_xlabel('# input identifiers (log scale)')
        ax4.set_ylabel('Throughput (it/s; log scale)')
        ax4.set_title('HLCA harmonization throughput scales across datasets')
        ax4.legend(loc='best', frameon=True)
        fig4.tight_layout()

        out_fig4 = MANUSCRIPT_FIGURES / 'fig_hlca_throughput_scaling.pdf'
        fig4.savefig(out_fig4, bbox_inches='tight')
        print('Saved:', out_fig4)
    else:
        print('No throughput timings available; skipping throughput scaling plot.')

    # 5) Shared feature-space structure: pairwise Jaccard heatmap (Ensembl 1→1 only)
    jaccard_path = HLCA_CACHE / 'hlca_jaccard_matrix.csv'
    if jaccard_path.exists():
        jacc = pd.read_csv(jaccard_path, index_col=0)

        fig5, ax5 = plt.subplots(1, 1, figsize=(8.5, 7.5))
        if sns is not None:
            sns.heatmap(jacc, ax=ax5, cmap='Blues', vmin=0, vmax=1, square=True, cbar_kws={'label': 'Jaccard'} )
        else:
            im = ax5.imshow(jacc.values, cmap='Blues', vmin=0, vmax=1)
            fig5.colorbar(im, ax=ax5, label='Jaccard')
            ax5.set_xticks(range(len(jacc.columns)))
            ax5.set_yticks(range(len(jacc.index)))
            ax5.set_xticklabels(jacc.columns, rotation=90)
            ax5.set_yticklabels(jacc.index)

        ax5.set_title('HLCA pairwise overlap after 1→1 harmonization (Ensembl backbone)')
        ax5.set_xlabel('Dataset')
        ax5.set_ylabel('Dataset')
        fig5.tight_layout()

        out_fig5 = MANUSCRIPT_FIGURES / 'fig_hlca_jaccard_heatmap.pdf'
        fig5.savefig(out_fig5, bbox_inches='tight')
        print('Saved:', out_fig5)
    else:
        print('No Jaccard matrix found; skipping heatmap:', jaccard_path)
